# Acoustic ID — Module 1: ID Generation & Database
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Accepts RTO operator input: plate number, owner name, contact, address
- Derives a unique 32-bit Acoustic ID via SHA-256 hashing of the plate number
- Appends a CRC-8 checksum to form the 40-bit transmission ID used by the signal encoder
- Persists all records in a SQLite database (`acoustic_id.db`)
- Exposes lookup functions for use by Module 3 (roadside decoder)

**Libraries:** `hashlib`, `sqlite3`, `re`, `os` — all Python standard library, no installation required.

## Step 1: Import Libraries

In [9]:
import hashlib
import sqlite3
import re
import os

## Step 2: CRC-8 Checksum

An 8-bit CRC is appended to every Acoustic ID before transmission.
The roadside receiver uses it to detect signal corruption before performing a database lookup.

- Polynomial: 0x07 (standard CRC-8)
- Input: 32-bit binary string (the Acoustic ID)
- Output: 8-bit binary string
- Combined transmission ID: 32 bits (ID) + 8 bits (CRC) = 40 bits total

In [2]:
def compute_crc8(data_bits: str) -> str:
    """
    Compute CRC-8 checksum for a binary string.

    Args:
        data_bits (str): Binary string of any length,
                         e.g. "00101101110010101011001101110101"

    Returns:
        str: 8-bit CRC as a binary string, e.g. "10110011"
    """
    padded = data_bits.zfill(((len(data_bits) + 7) // 8) * 8)
    byte_array = int(padded, 2).to_bytes(len(padded) // 8, byteorder="big")

    crc = 0
    for byte in byte_array:
        crc ^= byte
        for _ in range(8):
            if crc & 0x80:
                crc = (crc << 1) ^ 0x07
            else:
                crc <<= 1
            crc &= 0xFF

    return format(crc, "08b")

## Step 3: Number Plate Normalisation and Validation

Indian vehicle registration numbers follow the format: `SS DD LL NNNN`
- SS: 2-letter state code
- DD: 1-2 digit district code
- LL: 1-3 letter series
- NNNN: 4-digit number

Input may arrive with spaces or hyphens (e.g. `MH 12 AB 1234`, `KA-01-CD-5678`).
Both functions are called inside `generate_acoustic_id()` before any hashing.

In [3]:
def normalise_plate(plate: str) -> str:
    """
    Strip spaces and hyphens, convert to uppercase.

    Args:
        plate (str): Raw plate string as entered by RTO operator

    Returns:
        str: Normalised plate string, e.g. "MH12AB1234"
    """
    return plate.strip().upper().replace(" ", "").replace("-", "")


def validate_plate(plate: str) -> bool:
    """
    Validate that the normalised plate matches the Indian registration format.

    Args:
        plate (str): Normalised plate string

    Returns:
        bool: True if format is valid
    """
    pattern = r"^[A-Z]{2}[0-9]{1,2}[A-Z]{1,3}[0-9]{4}$"
    return bool(re.match(pattern, plate))

## Step 4: Acoustic ID Generation

**Algorithm:**
1. Normalise and validate the plate string
2. Apply SHA-256 — produces a 256-bit digest
3. Extract the first 32 bits as the Acoustic ID
4. Collision check: if that 32-bit slice is already assigned, slide the window
   by 32 bits and retry (up to 7 windows within the 256-bit hash)
5. Compute and append CRC-8 to produce the 40-bit transmission ID

**Properties:**
- Deterministic: the same plate always yields the same ID
- Collision-resistant: SHA-256 makes accidental collisions negligible at India's
  vehicle scale (~320 million vehicles vs 4.29 billion possible 32-bit IDs)
- Non-reversible: the plate cannot be reconstructed from the ID alone

In [4]:
def generate_acoustic_id(plate: str, existing_ids: set = None) -> dict:
    """
    Generate a unique 32-bit Acoustic ID from a vehicle number plate.

    Args:
        plate        (str) : Raw or normalised plate string
        existing_ids (set) : Set of 32-bit binary ID strings already present
                             in the database, used for collision avoidance.
                             Pass None or an empty set if calling in isolation.

    Returns:
        dict: {
            "plate"           : str  -- normalised plate,
            "acoustic_id"     : str  -- 32-bit binary string,
            "acoustic_id_hex" : str  -- 8-character hex representation,
            "crc8"            : str  -- 8-bit binary CRC,
            "transmission_id" : str  -- 40-bit binary string (ID + CRC)
        }

    Raises:
        ValueError    : If the plate format is invalid
        OverflowError : If all 32-bit windows in the SHA-256 hash are exhausted
    """
    if existing_ids is None:
        existing_ids = set()

    norm_plate = normalise_plate(plate)

    if not validate_plate(norm_plate):
        raise ValueError(f"Invalid plate format: '{norm_plate}'")

    hash_bytes = hashlib.sha256(norm_plate.encode("utf-8")).digest()
    full_hash_bits = "".join(format(b, "08b") for b in hash_bytes)  # 256 bits

    acoustic_id = None
    for offset in range(0, 256 - 32 + 1, 32):
        candidate = full_hash_bits[offset:offset + 32]
        if candidate not in existing_ids:
            acoustic_id = candidate
            break

    if acoustic_id is None:
        raise OverflowError(
            f"All 32-bit windows of SHA-256 hash for '{norm_plate}' are already "
            "assigned. This is statistically near-impossible under normal operating "
            "conditions."
        )

    crc = compute_crc8(acoustic_id)
    transmission_id = acoustic_id + crc
    acoustic_id_hex = format(int(acoustic_id, 2), "08X")

    return {
        "plate"           : norm_plate,
        "acoustic_id"     : acoustic_id,
        "acoustic_id_hex" : acoustic_id_hex,
        "crc8"            : crc,
        "transmission_id" : transmission_id
    }

## Step 5: Database Setup

Creates `acoustic_id.db` with a single `vehicles` table.

| Column | Source | Notes |
|---|---|---|
| `plate_number` | RTO operator input | Primary key |
| `owner_name` | RTO operator input | Registered owner |
| `owner_contact` | RTO operator input | Mobile number for challan notification |
| `owner_address` | RTO operator input | Registered address |
| `acoustic_id` | Auto-derived | 32-bit binary ID |
| `acoustic_id_hex` | Auto-derived | Hex form for display and logging |
| `transmission_id` | Auto-derived | 40-bit ID + CRC-8, used by signal encoder |
| `module_serial` | System default | `UNASSIGNED` until hardware is fitted |
| `ecu_key_hash` | System default | `UNPAIRED` until module is installed |
| `status` | System default | `active` on registration |
| `issued_date` | Auto-timestamp | Date of registration |
| `last_verified` | Auto-timestamp | Updated on each successful ECU handshake |



In [5]:
def setup_database(db_path: str = "acoustic_id.db") -> sqlite3.Connection:
    """
    Initialise the Acoustic ID SQLite database and create the vehicles table
    if it does not already exist.

    Args:
        db_path (str): File path for the SQLite database.
                       Defaults to "acoustic_id.db" in the current directory.

    Returns:
        sqlite3.Connection: Open connection with row_factory set to sqlite3.Row
    """
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    conn.execute("""
        CREATE TABLE IF NOT EXISTS vehicles (
            plate_number     TEXT PRIMARY KEY,
            owner_name       TEXT NOT NULL,
            owner_contact    TEXT NOT NULL,
            owner_address    TEXT NOT NULL,
            acoustic_id      TEXT UNIQUE NOT NULL,
            acoustic_id_hex  TEXT UNIQUE NOT NULL,
            transmission_id  TEXT UNIQUE NOT NULL,
            module_serial    TEXT NOT NULL DEFAULT 'UNASSIGNED',
            ecu_key_hash     TEXT NOT NULL DEFAULT 'UNPAIRED',
            status           TEXT NOT NULL DEFAULT 'active'
                             CHECK(status IN ('active', 'tampered', 'stolen', 'deactivated')),
            issued_date      TEXT NOT NULL DEFAULT (datetime('now')),
            last_verified    TEXT NOT NULL DEFAULT (datetime('now'))
        )
    """)
    conn.commit()
    return conn


# Initialise connection — all subsequent cells use this conn object
conn = setup_database()

## Step 6: Vehicle Registration

`register_vehicle()` is the single-entry registration function.
`register_vehicles_batch()` wraps it for bulk RTO data import.

**Input fields (RTO operator provides):**
- `plate_number`, `owner_name`, `owner_contact`, `owner_address`

**Idempotency:** if a plate already exists, the existing record is returned without
modification. The Acoustic ID is never regenerated for a plate that has been assigned one.

**Batch input format:**
```python
vehicles = [
    {
        'plate_number'  : 'MH12AB1234',
        'owner_name'    : 'Name',
        'owner_contact' : '9876543210',
        'owner_address' : 'Address'
    },
    ...
]
register_vehicles_batch(vehicles, conn)
```

In [6]:
def register_vehicle(
    plate_number  : str,
    owner_name    : str,
    owner_contact : str,
    owner_address : str,
    conn          : sqlite3.Connection
) -> dict:
    """
    Register a single vehicle and persist its Acoustic ID to the database.

    Args:
        plate_number  (str): Vehicle registration number (raw or normalised)
        owner_name    (str): Full name of registered owner
        owner_contact (str): Mobile number
        owner_address (str): Registered address
        conn (sqlite3.Connection): Active database connection

    Returns:
        dict: Full database record for the vehicle

    Raises:
        ValueError: If the plate format is invalid
    """
    norm_plate = normalise_plate(plate_number)

    # Return existing record unchanged — never regenerate an assigned ID
    row = conn.execute(
        "SELECT * FROM vehicles WHERE plate_number = ?", (norm_plate,)
    ).fetchone()
    if row:
        return dict(row)

    # Load existing IDs for collision avoidance
    existing_ids = {
        r[0] for r in conn.execute("SELECT acoustic_id FROM vehicles").fetchall()
    }

    id_data = generate_acoustic_id(norm_plate, existing_ids)

    conn.execute(
        """
        INSERT INTO vehicles
            (plate_number, owner_name, owner_contact, owner_address,
             acoustic_id, acoustic_id_hex, transmission_id)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (
            id_data["plate"],
            owner_name,
            owner_contact,
            owner_address,
            id_data["acoustic_id"],
            id_data["acoustic_id_hex"],
            id_data["transmission_id"]
        )
    )
    conn.commit()

    return dict(
        conn.execute(
            "SELECT * FROM vehicles WHERE plate_number = ?", (id_data["plate"],)
        ).fetchone()
    )


def register_vehicles_batch(
    vehicles : list,
    conn     : sqlite3.Connection
) -> list:
    """
    Register multiple vehicles in a single call.

    Args:
        vehicles (list of dict): Each dict must contain:
            "plate_number", "owner_name", "owner_contact", "owner_address"
        conn (sqlite3.Connection): Active database connection

    Returns:
        list of dict: Registration records for all vehicles

    Raises:
        KeyError: If a required field is missing from any entry
    """
    required = {"plate_number", "owner_name", "owner_contact", "owner_address"}
    records = []

    for entry in vehicles:
        missing = required - entry.keys()
        if missing:
            raise KeyError(
                f"Missing required fields {missing} in entry: {entry}"
            )
        records.append(
            register_vehicle(
                plate_number  = entry["plate_number"],
                owner_name    = entry["owner_name"],
                owner_contact = entry["owner_contact"],
                owner_address = entry["owner_address"],
                conn          = conn
            )
        )

    return records

## Step 7: Lookup Functions

| Function | Caller | Purpose |
|---|---|---|
| `lookup_by_plate()` | RTO portal / admin | Retrieve Acoustic ID given a plate |
| `lookup_by_acoustic_id()` | Module 3 | Identify vehicle after decoding ultrasonic signal |
| `lookup_by_transmission_id()` | Module 3 | CRC validation then ID lookup — primary production path |

`lookup_by_transmission_id()` is the function Module 3 calls in production.
It rejects corrupted signals at the CRC stage before touching the database.
A failed CRC returns `None`; Module 3 is responsible for logging the event.

In [7]:
def lookup_by_plate(plate: str, conn: sqlite3.Connection) -> dict | None:
    """
    Retrieve vehicle record by number plate.

    Args:
        plate (str): Raw or normalised plate string
        conn  (sqlite3.Connection): Active database connection

    Returns:
        dict: Vehicle record, or None if plate is not registered
    """
    row = conn.execute(
        "SELECT * FROM vehicles WHERE plate_number = ?",
        (normalise_plate(plate),)
    ).fetchone()
    return dict(row) if row else None


def lookup_by_acoustic_id(acoustic_id: str, conn: sqlite3.Connection) -> dict | None:
    """
    Retrieve vehicle record by 32-bit Acoustic ID.

    Called by Module 3 after OOK demodulation and CRC validation.

    Args:
        acoustic_id (str): 32-bit binary string decoded from the ultrasonic signal
        conn        (sqlite3.Connection): Active database connection

    Returns:
        dict: Vehicle record, or None if ID is not in the database
    """
    row = conn.execute(
        "SELECT * FROM vehicles WHERE acoustic_id = ?",
        (acoustic_id,)
    ).fetchone()
    return dict(row) if row else None


def lookup_by_transmission_id(transmission_id: str, conn: sqlite3.Connection) -> dict | None:
    """
    Validate CRC then retrieve vehicle record by 40-bit transmission ID.

    Primary lookup entry point for Module 3. A signal that fails CRC
    is discarded before any database query is made.

    Args:
        transmission_id (str): 40-bit binary string (32-bit ID + 8-bit CRC)
        conn            (sqlite3.Connection): Active database connection

    Returns:
        dict: Vehicle record if CRC passes and ID is found, otherwise None

    Raises:
        ValueError: If transmission_id length is not exactly 40 bits
    """
    if len(transmission_id) != 40:
        raise ValueError(
            f"transmission_id must be 40 bits, received {len(transmission_id)}"
        )

    acoustic_id  = transmission_id[:32]
    received_crc = transmission_id[32:]
    computed_crc = compute_crc8(acoustic_id)

    if received_crc != computed_crc:
        return None  # Corrupted signal — caller is responsible for logging

    return lookup_by_acoustic_id(acoustic_id, conn)

## Step 8: Database Viewer

Utility function for inspection at the RTO console or during debugging.
Not part of the enforcement pipeline — call manually as needed.

In [8]:
def show_database(conn: sqlite3.Connection) -> None:
    """
    Print all registered vehicles in a tabular format.

    Columns displayed: sequence number, plate, acoustic ID (hex),
    acoustic ID (binary), owner name, status, issued date.

    Args:
        conn (sqlite3.Connection): Active database connection
    """
    rows = conn.execute(
        """
        SELECT plate_number, acoustic_id_hex, acoustic_id,
               owner_name, status, issued_date
        FROM   vehicles
        ORDER  BY issued_date ASC
        """
    ).fetchall()

    if not rows:
        print("No vehicles registered.")
        return

    col_widths = {
        "num"    : 4,
        "plate"  : 12,
        "hex"    : 10,
        "binary" : 34,
        "owner"  : 22,
        "status" : 12,
        "date"   : 20
    }

    header = (
        f"{'#':<{col_widths['num']}}"  
        f"  {'Plate':<{col_widths['plate']}}"  
        f"  {'ID (Hex)':<{col_widths['hex']}}"  
        f"  {'Acoustic ID (Binary)':<{col_widths['binary']}}"  
        f"  {'Owner':<{col_widths['owner']}}"  
        f"  {'Status':<{col_widths['status']}}"  
        f"  Issued"
    )
    print(header)
    print("-" * len(header))

    for i, row in enumerate(rows, 1):
        print(
            f"{i:<{col_widths['num']}}"  
            f"  {row[0]:<{col_widths['plate']}}"  
            f"  {row[1]:<{col_widths['hex']}}"  
            f"  {row[2]:<{col_widths['binary']}}"  
            f"  {row[3]:<{col_widths['owner']}}"  
            f"  {row[4]:<{col_widths['status']}}"  
            f"  {row[5]}"
        )

    total = len(rows)
    capacity = 4_294_967_296
    print(f"\nRegistered : {total:,}")
    print(f"Capacity   : {capacity:,}  (32-bit)")
    print(f"Utilised   : {total / capacity * 100:.8f}%")

## Step 9: Load Vehicle Data from Files

This section allows loading vehicle data from JSON or Excel files.
The data should be in the format: list of dictionaries with keys "plate_number", "owner_name", "owner_contact", "owner_address".

For JSON: [{"plate_number": "MH12AB1234", "owner_name": "John Doe", ...}, ...]

For Excel: columns named plate_number, owner_name, owner_contact, owner_address.

In [ ]:
import json
import pandas as pd


def load_vehicles_from_json(file_path: str) -> list:
    """
    Load vehicle data from a JSON file.

    Args:
        file_path (str): Path to the JSON file containing a list of vehicle dictionaries.

    Returns:
        list: List of vehicle dictionaries.
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data


def load_vehicles_from_excel(file_path: str) -> list:
    """
    Load vehicle data from an Excel file.

    Assumes the Excel file has columns: plate_number, owner_name, owner_contact, owner_address.

    Args:
        file_path (str): Path to the Excel file.

    Returns:
        list: List of vehicle dictionaries.
    """
    df = pd.read_excel(file_path)
    return df.to_dict('records')


# Automated loading and registration
file_path = input("Enter the path to your vehicle data file (JSON or Excel): ").strip()

if file_path.endswith('.json'):
    vehicles_data = load_vehicles_from_json(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    vehicles_data = load_vehicles_from_excel(file_path)
else:
    print("Unsupported file type. Please use .json, .xlsx, or .xls")
    vehicles_data = []

if vehicles_data:
    registered = register_vehicles_batch(vehicles_data, conn)
    print(f"Registered {len(registered)} vehicles.")
    show_database(conn)
else:
    print("No data loaded.")